# 🏛️ BharatCRS V7 - Transformers Multi-Task Training
This notebook trains the **6-domain, 20-issue** civic classifier using the Muril backbone.

### 📋 Instructions:
1. **Generate Data**: Run `prepare_v7.py` locally to get `bharatcrs_v7_clean.csv`.
2. **Upload**: Upload `bharatcrs_v7_clean.csv` and `label_maps_v7.json` to this Colab session.
3. **Train**: Run all cells. This will export a `bharatcrs_v7.onnx` model at the end.

In [ ]:
!pip install transformers datasets torch onnx onnxruntime pandas numpy

In [ ]:
import torch
import pandas as pd
import numpy as np
import json
import os
from transformers import AutoTokenizer, AutoModel, AutoConfig
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# --- CONFIG ---
MODEL_NAME = 'google/muril-base-cased'
MAX_LEN = 256
BATCH_SIZE = 8
EPOCHS = 15
LR = 1e-5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Using device: {DEVICE}")

In [ ]:
class CivicDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_len):
        self.df = pd.read_csv(csv_path)
        self.tokenizer = tokenizer
        self.max_len = max_len
        
        # Label Mappings (Must match the generator)
        self.domain_labels = sorted(self.df['primary_domain'].unique().tolist())
        self.issue_labels = sorted(self.df['issue_type'].unique().tolist())
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, item):
        row = self.df.iloc[item]
        text = str(row['raw_text'])
        
        inputs = self.tokenizer(
            text, 
            max_length=self.max_len, 
            padding='max_length', 
            truncation=True, 
            return_tensors='pt'
        )
        
        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'domain': torch.tensor(self.domain_labels.index(row['primary_domain'])),
            'issue': torch.tensor(self.issue_labels.index(row['issue_type'])),
            'severity': torch.tensor(float(row['severity_level'] - 1) / 9.0, dtype=torch.float),
            'safety': torch.tensor(1.0 if row['public_safety_flag'] else 0.0, dtype=torch.float),
            'vuln': torch.tensor(1.0 if row.get('vulnerable_population_flag') else 0.0, dtype=torch.float)
        }

In [ ]:
class MultiTaskCivicClassifier(nn.Module):
    def __init__(self, n_domains, n_issues):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        hidden_size = self.bert.config.hidden_size
        
        self.domain_head = nn.Linear(hidden_size, n_domains)
        self.issue_head = nn.Linear(hidden_size, n_issues)
        self.severity_head = nn.Sequential(nn.Linear(hidden_size, 1), nn.Sigmoid())
        self.safety_head = nn.Linear(hidden_size, 1)
        self.vuln_head = nn.Linear(hidden_size, 1)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output
        
        return (
            self.domain_head(pooled), 
            self.issue_head(pooled), 
            self.severity_head(pooled), 
            self.safety_head(pooled), 
            self.vuln_head(pooled)
        )

print("Training Logic defined.")

In [ ]:
# --- EXPORT TO ONNX ---
def export_onnx(model, n_domains, n_issues):
    model.eval()
    dummy_input = torch.ones(1, MAX_LEN, dtype=torch.long).to(DEVICE)
    dummy_mask = torch.ones(1, MAX_LEN, dtype=torch.long).to(DEVICE)
    
    torch.onnx.export(
        model, 
        (dummy_input, dummy_mask), 
        'bharatcrs_v7.onnx', 
        input_names=['input_ids', 'attention_mask'],
        output_names=['domain_logits', 'issue_logits', 'severity_pred', 'safety_logit', 'vulnerable_logit'],
        dynamic_axes={'input_ids': {0: 'batch_size'}, 'attention_mask': {0: 'batch_size'}},
        opset_version=12
    )
    print("ONNX Export Complete!")